In [1]:
from urllib.request import urlopen, Request
from urllib.parse import urlencode, urlparse
import xml.etree.ElementTree  as ET
import requests, json
from IPython.display import display, HTML
# import rasterio as rio

URL = r"https://geo.api.vlaanderen.be/el-dtm/wcs"

In [41]:

url= "https://www.dov.vlaanderen.be/geoserver/pfas/wfs?service=wfs&request=getcapabilities"

xdata=  requests.get(url).text
result = ET.fromstring(xdata)

wfs_ns ={'wfs': 'http://www.opengis.net/wfs/2.0'}
result.findall( ".//wfs:FeatureType" , wfs_ns)
len( [n for n in range(100, 999+1, 100)] ) , 999 //100

(9, 9)

In [2]:
ns = {
    'wcs': "http://www.opengis.net/wcs/2.0",
    'ows': "http://www.opengis.net/ows/2.0",
    'gml': "http://www.opengis.net/gml/3.2"
}

In [4]:
meta_id= 'e68b0395-0f83-4de4-b22e-aa8151edcba7'
url = f"https://metadata.vlaanderen.be/srv/dut/csw?service=CSW&version=2.0.2&request=GetRecordById&id={meta_id}&ElementSetName=full"

xdata=  requests.get(url).text
mdata = ET.fromstring(xdata)

In [10]:
print( xdata )

<?xml version="1.0" encoding="UTF-8"?>
<csw:GetRecordByIdResponse xmlns:csw="http://www.opengis.net/cat/csw/2.0.2">
  <csw:Record xmlns:gmx="http://www.isotc211.org/2005/gmx" xmlns:ows="http://www.opengis.net/ows" xmlns:dct="http://purl.org/dc/terms/" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:geonet="http://www.fao.org/geonetwork">
    <dc:identifier>e68b0395-0f83-4de4-b22e-aa8151edcba7</dc:identifier>
    <dc:date>2025-11-18</dc:date>
    <dc:title>Landbouwgebruikspercelen LV, 2024</dc:title>
    <dc:type>dataset</dc:type>
    <dc:subject>Regionaal</dc:subject>
    <dc:subject>Faciliteiten voor landbouw en aquacultuur</dc:subject>
    <dc:subject>geografie</dc:subject>
    <dc:subject>Kosteloos</dc:subject>
    <dc:subject>Toegevoegd GDI-Vl</dc:subject>
    <dc:subject>Vlaamse Open data</dc:subject>
    <dc:subject>Geografische gegevens</dc:subject>
    <dc:subject>Lijst M&amp;R INSPIRE</dc:subject>
    <dc:subject>landbouw</dc:subject>
    <dc:subject>land parcel identificati

In [28]:
def unique_set(data, key):
    seen = set()
    result = []
    for obj in data:
        if obj[key] not in seen:
            seen.add(obj[key])
            result.append(obj)
    return result

unique_set([{'n': 1},{'z': 1, 'n': 0},{'n': 1}], 'n')

[{'n': 1}, {'z': 1, 'n': 0}]

In [ ]:
links = mdata.find("{http://www.opengis.net/cat/csw/2.0.2}Record").findall('{http://purl.org/dc/elements/1.1/}URI')
from urllib.parse import urlparse

for link in links:
    l_url = urlparse( link.text  )
    if not l_url:
        continue
    print( link.attrib.get("name", 'Geen naam') )
    



LbGebrPerc2024
Landbgebrperc:Lbgebrperc
Landbgebrperc:Lbgebrperc
Landbgebrperc
Lbgebrperc
Landbouwgebruikspercelen LV, 2024
Voorbeeldweergave


In [3]:
req = Request(
    URL + "?request=getcapabilities", 
    headers={
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/35.0.1916.47 Safari/537.36'
    }
)

resp = urlopen(req)
capa = resp.read().decode()

In [5]:
root = ET.fromstring(capa)

for cov in  root.findall('.//wcs:CoverageSummary', namespaces=ns):
    Title = cov.find("./ows:Title", ns)
    covId = cov.find("./wcs:CoverageId", ns) 
    print(Title.text , " -> ", covId.text )

Digitaal Hoogtemodel Vlaanderen II, DTM, raster, 1 m  ->  EL.GridCoverage.DTM


In [ ]:

req = Request(
URL+"?service=WCS&version=2.0.1&request=describecoverage&coverageid=EL.GridCoverage.DTM", 
headers={
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/35.0.1916.47 Safari/537.36'
    }
)
resp = urlopen(req)
descript = resp.read().decode()
print(descript)

In [4]:
BBOX = [2.5170092434134, 50.669499282716, 5.9444417082211, 51.51507303755]

params = {
    "SERVICE":"WCS",
    "VERSION":"2.0.1",
    "request":"GetCoverage",
    "CRS":"EPSG:31370",
    "subset":"y,http://www.opengis.net/def/crs/EPSG/0/4258(50.7461,50.9108)",
    "subset":"x,http://www.opengis.net/def/crs/EPSG/0/4258(4.2548,4.5418)",
    "scalefactor":"50",
    "COVERAGEID":"EL.GridCoverage.DTM",
    "FORMAT":"image/tiff", 
    "RESPONSE_CRS": "EPSG:31370"
}

In [25]:
req = Request(
    URL + "?" + urlencode(params), 
    headers={
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/35.0.1916.47 Safari/537.36'
    }
)
print( URL + "?" + urlencode(params) )
resp = urlopen(req)

https://geo.api.vlaanderen.be/el-dtm/wcs?SERVICE=WCS&VERSION=2.0.1&request=GetCoverage&CRS=EPSG%3A31370&subset=x%2Chttp%3A%2F%2Fwww.opengis.net%2Fdef%2Fcrs%2FEPSG%2F0%2F4258%284.2548%2C4.5418%29&scalefactor=50&COVERAGEID=EL.GridCoverage.DTM&FORMAT=image%2Ftiff&RESPONSE_CRS=EPSG%3A31370


In [ ]:
gml = ''
resp.readline() #: first line
for line in resp:
   lined = line.decode(encoding="utf8")
   if '--wcs' in lined: break
   gml +=  lined 

for line in resp:
   lined = line.decode( encoding="ascii")
   if  'Content-Disposition' in lined: break

with open("test.tif", mode="wb") as f:
   for line in resp:
      f.write(line)
print(gml)

In [28]:
ras = rio.open(r'D:\work\geopunt4Qgis\docs\test.tif') 
print(ras.width, ras.height)
ras.close()

5170 1357


In [1]:
import requests


URL = "https://geo.api.vlaanderen.be/el-dtm/wcs"

data_post="""<?xml version="1.0" encoding="UTF-8"?>
<wcs:GetCapabilities
 xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
 xmlns:wcs="http://www.opengis.net/wcs/2.0" 
 xmlns:ows="http://www.opengis.net/ows/2.0" 
 xsi:schemaLocation= "http://schemas.opengis.net/wcs/2.0 ../wcsAll.xsd" service="WCS">
 <ows:AcceptVersions>
 <ows:Version>2.0.1</ows:Version>
 </ows:AcceptVersions>
</wcs:GetCapabilities>"""

headers = {
  'Content-Type': 'application/xml',
  'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; rv:91.0) Gecko/20100101 Firefox/91.0'
}

_r = requests.post(URL, data=data_post, headers=headers)
print( _r.content.decode() )

<?xml version="1.0" encoding="UTF-8"?>
<wcs:Capabilities xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
 xmlns:ows="http://www.opengis.net/ows/2.0"
 xmlns:wcs="http://www.opengis.net/wcs/2.0"
 xmlns:gml="http://www.opengis.net/gml/3.2"
 xmlns:crs="http://www.opengis.net/wcs/service-extension/crs/1.0"
 xmlns:int="http://www.opengis.net/WCS_service-extension_interpolation/1.0"
 xmlns="http://www.opengis.net/ows/2.0"
 xmlns:xlink="http://www.w3.org/1999/xlink"
 xsi:schemaLocation="http://www.opengis.net/wcs/2.0 http://schemas.opengis.net/wcs/2.0/wcsAll.xsd" version="2.0.1">
  <ows:ServiceIdentification>
    <ows:Title>el</ows:Title>
  <ows:ServiceType>OGC WCS</ows:ServiceType>
  <ows:ServiceTypeVersion>2.0.1</ows:ServiceTypeVersion>
  <ows:ServiceTypeVersion>1.1.2</ows:ServiceTypeVersion>
  <ows:ServiceTypeVersion>1.1.1</ows:ServiceTypeVersion>
  <ows:ServiceTypeVersion>1.1.0</ows:ServiceTypeVersion>
  <ows:ServiceTypeVersion>1.0.0</ows:ServiceTypeVersion>
  <ows:Profile>http://www

In [8]:
data_post="""<?xml version="1.0" encoding="UTF-8"?> 
<wcs:GetCoverage
 xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
 xmlns:wcs="http://www.opengis.net/wcs/2.0" 
 xmlns:ows="http://www.opengis.net/ows/2.0" 
 xmlns:gml="http://www.opengis.net/gml/3.2" 
 xsi:schemaLocation= "http://schemas.opengis.net/wcs/2.0 ../wcsAll.xsd"
  service="WCS" version="2.0.1"> 
  <wcs:CoverageId>EL.GridCoverage.DTM</wcs:CoverageId> 
  <wcs:trimDimension> 
    <wcs:dimension>x</wcs:dimension> 
    <wcs:trimLow>50.7461</wcs:trimLow> 
    <wcs:trimHigh>50.9108</wcs:trimHigh> 
  </wcs:trimDimension> 
  <wcs:trimDimension> 
    <wcs:dimension>x</wcs:dimension> 
    <wcs:trimLow>4.2548</wcs:trimLow> 
    <wcs:trimHigh>4.5418</wcs:trimHigh> 
  </wcs:trimDimension>
</wcs:GetCoverage>"""

url = 'https://geo.api.vlaanderen.be/el-dtm/wcs'
headers={
    
       'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/35.0.1916.47 Safari/537.36'
    }


_r = requests.post(url, data=data_post, headers=headers )
print(_r.text)

<html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>
<hr><center>Microsoft-Azure-Application-Gateway/v2</center>
</body>
</html>
<!-- a padding to disable MSIE and Chrome friendly error page -->
<!-- a padding to disable MSIE and Chrome friendly error page -->
<!-- a padding to disable MSIE and Chrome friendly error page -->
<!-- a padding to disable MSIE and Chrome friendly error page -->
<!-- a padding to disable MSIE and Chrome friendly error page -->
<!-- a padding to disable MSIE and Chrome friendly error page -->



In [18]:
from osgeo import gdal
url= r'WCS:https://geo.api.vlaanderen.be/el-dtm/wcs?VERSION=1.1.0&coverage=EL.GridCoverage.DTM'
dataset = gdal.Open( url , gdal.GA_ReadOnly)
dataset.GetGeoTransform()

(51.559484250526744,
 -1.3737840783946172e-05,
 0.0,
 2.461217912928196,
 0.0,
 1.3737840783946142e-05)

In [34]:
dataset.RasterXSize//100 , dataset.RasterYSize //100

(2584, 678)

In [36]:
dataset.ReadAsArray(xsize= dataset.RasterXSize//100, ysize= dataset.RasterYSize //100, callback=print)

1.0 Downloading ... None


In [11]:

band = dataset.GetRasterBand(1)
band.ComputeRasterMinMax(True)

scanline = band.ReadRaster(xoff=0, yoff=0,
                        xsize=band.XSize, ysize=1,
                        buf_xsize=band.XSize, buf_ysize=1,
                        buf_type=gdal.GDT_Float32)
scanline

In [8]:
f =[[4,54],[1,2],4,5]
f[1:]

[[1, 2], 4, 5]

In [5]:
import requests

def fetchElevaton(LineString, srs=31370, samples=20 ):
    baseUri = 'https://dhm.agiv.be/api/elevation/v1/DHMV2'
    locations = "|".join( ["{},{}".format(*f) for f in LineString ] )
    data = {}
    data["SrsIn"] = srs
    data["SrsOut"] = srs 
    data["Locations"] = locations
    data["Samples"] = samples
    resp = requests.get(baseUri, params=data )
    return resp.json()

In [6]:
ln = [[152898.7,212780.2],[153071.1,211220.9]]

fetchElevaton(ln)

[[0.0, 152898.7, 212780.2, 6.29],
 [82.56850173779661, 152907.77368421055, 212698.1315789474, 7.25],
 [165.13700347559322, 152916.84736842106, 212616.06315789474, 8.21],
 [247.7055052133898, 152925.92105263157, 212533.9947368421, 8.43],
 [330.27400695118644, 152934.9947368421, 212451.9263157895, 7.93],
 [412.84250868898306, 152944.06842105265, 212369.85789473687, 8.54],
 [495.4110104267797, 152953.14210526316, 212287.7894736842, 8.07],
 [577.9795121645762, 152962.21578947367, 212205.72105263156, 8.69],
 [660.5480139023729, 152971.2894736842, 212123.65263157897, 7.09],
 [743.1165156401695, 152980.36315789475, 212041.58421052631, 7.41],
 [825.6850173779661, 152989.4368421053, 211959.51578947366, 7.56],
 [908.2535191157627, 152998.5105263158, 211877.44736842107, 7.23],
 [990.8220208535594, 153007.58421052631, 211795.37894736842, 7.44],
 [1073.3905225913559, 153016.65789473685, 211713.31052631576, 8.28],
 [1155.9590243291525, 153025.73157894737, 211631.24210526314, 7.4],
 [1238.52752606694

In [1]:
import numpy as np

e= np.array( [[0.0, 152898.7, 212780.2, 6.29],
 [82.56850173779661, 152907.77368421055, 212698.1315789474, 7.25],
 [165.13700347559322, 152916.84736842106, 212616.06315789474, 8.21],
 [247.7055052133898, 152925.92105263157, 212533.9947368421, 8.43],
 [330.27400695118644, 152934.9947368421, 212451.9263157895, 7.93],
 [412.84250868898306, 152944.06842105265, 212369.85789473687, 8.54],
 [495.4110104267797, 152953.14210526316, 212287.7894736842, 8.07],
 [577.9795121645762, 152962.21578947367, 212205.72105263156, 8.69],
 [660.5480139023729, 152971.2894736842, 212123.65263157897, 7.09],
 [743.1165156401695, 152980.36315789475, 212041.58421052631, 7.41],
 [825.6850173779661, 152989.4368421053, 211959.51578947366, 7.56],
 [908.2535191157627, 152998.5105263158, 211877.44736842107, 7.23],
 [990.8220208535594, 153007.58421052631, 211795.37894736842, 7.44],
 [1073.3905225913559, 153016.65789473685, 211713.31052631576, 8.28],
 [1155.9590243291525, 153025.73157894737, 211631.24210526314, 7.4],
 [1238.5275260669491, 153034.8052631579, 211549.17368421052, 7.7],
 [1321.0960278047457, 153043.87894736842, 211467.1052631579, 6.73],
 [1403.6645295425424, 153052.95263157896, 211385.03684210527, 6.78],
 [1486.233031280339, 153062.0263157895, 211302.96842105262, 4.21],
 [1568.8015330181356, 153071.1, 211220.9, 5.82]] )


In [66]:
from urllib.parse import unquote, urlencode, urlparse
from urllib.request import urlopen
from pathlib import Path
import json

url = "https://geo.api.vlaanderen.be/ef/ogc/features/v1/collections/b/?f=application%2Fjson"
p =urlparse(url)


p.path
parts =  [p for p in p.path.split('/') if p != '']

if 'collections' in parts:
    idx = parts.index('collections')
    path_until = '/'.join(parts[:idx +1])
    result = f"{p.scheme}://{p.netloc}/{path_until}"
else:
    result = f"{p.scheme}://{p.netloc}/{p.path}"
result

'https://geo.api.vlaanderen.be/ef/ogc/features/v1/collections'

In [69]:
json.load( urlopen( result ) )

{'links': [{'href': 'https://geo.api.vlaanderen.be/ef/ogc/features/v1/collections/?f=application%2Fjson',
   'rel': 'self',
   'type': 'application/json',
   'title': 'This document'},
  {'href': 'https://geo.api.vlaanderen.be/ef/ogc/features/v1/collections/?f=application%2Fyaml',
   'rel': 'alternate',
   'type': 'application/yaml',
   'title': 'This document as application/yaml'},
  {'href': 'https://geo.api.vlaanderen.be/ef/ogc/features/v1/collections/?f=text%2Fhtml',
   'rel': 'alternate',
   'type': 'text/html',
   'title': 'This document as text/html'}],
 'collections': [{'id': 'EnvironmentalMonitoringFacility',
   'title': 'Milieubewakingsvoorziening',
   'description': 'Een aan geografische coördinaten gerelateerd object dat rechtstreeks gegevens verzamelt of verwerkt over objecten waarvan de eigenschappen (bv. fysische, chemische, biologische of andere aspecten van milieuomstandigheden) met geregelde tussenpozen worden geobserveerd of gemeten. Een milieubewakingsvoorziening ka

In [76]:
url = "https://geo.api.vlaanderen.be/ef/ogc/features/v1/collections/b/test?f=application%2Fjson"
def get_ogc_api_collections( url):
    p = urlparse(url)
    parts = p.path.split('/')

    if 'collections' in parts:
        idx = parts.index('collections')
        path_until = '/'.join(parts[:idx+1 ])
        baseurl = f"{p.scheme}://{p.netloc}/{path_until}"
    else:
        baseurl = f"{p.scheme}://{p.netloc}/{p.path}/collections"
    default_name = unquote( Path(baseurl).stem )
    resp = urlopen( baseurl  )
    collections = json.load(resp)
    lyrNames = [ (c['id'] , c['title'] , c['description'])
            for c in collections["collections"]
        ]
    if default_name in [c[0] for c in lyrNames]:
        lyrNames = [ next((r for r in lyrNames if r[0] == default_name) ) ]

    return lyrNames

In [77]:
get_ogc_api_collections(url)

[('EnvironmentalMonitoringFacility',
  'Milieubewakingsvoorziening',
  'Een aan geografische coördinaten gerelateerd object dat rechtstreeks gegevens verzamelt of verwerkt over objecten waarvan de eigenschappen (bv. fysische, chemische, biologische of andere aspecten van milieuomstandigheden) met geregelde tussenpozen worden geobserveerd of gemeten. Een milieubewakingsvoorziening kan ook plaats bieden aan andere milieubewakingsvoorzieningen.'),
 ('EnvironmentalMonitoringNetwork',
  'Milieubewakingsnetwerk',
  'Administratieve of organisatorische ordening van EnvironmentalMonitoringFacilities die op eendere wijze worden beheerd voor een specifiek doeleinde en daarbij op een specifiek gebied zijn gericht. Elk netwerk functioneert volgens gemeenschappelijke regels die de coherentie van de observaties dienen te waarborgen, met name met het oog op de werking van EnvironmentalMonitoringFacilities, de vaststelling van voorgeschreven parameters, meetmethoden en het meetkader.')]

In [4]:
a = np.arange(5)
b = np.zeros(5)
np.hstack(
   (a.reshape((-1,1)),
    b.reshape((-1,1)) )
)

array([[0., 0.],
       [1., 0.],
       [2., 0.],
       [3., 0.],
       [4., 0.]])

In [2]:
import pathlib

In [4]:
pathlib.Path(r'P:\archivering\publiek_domein').as_uri()

'file:///P:/archivering/publiek_domein'

In [3]:
_KEY = "9f1c0c85-38ab-473c-a01b-069f04bf6386"
_BASEURL = "https://datavindplaats.api.vlaanderen.be"

headers ={'x-api-key': _KEY } 

def getItemData(self, identifier:str) -> dict:
    url = self.baseUrl + '/v1/catalogrecords/' + identifier
    resp = requests.get( url, headers=headers)
    return json.loads(resp.text)

def findItems( q:str, c:int=100, offset:int=0, taxonomy:str=None):
    url = _BASEURL + '/v1/catalogrecords'
    params = {'q': q, 'limit': c, 'offset': offset} 
    if taxonomy: 
        params['taxonomy']= taxonomy
    resp =  requests.get( url, params=params, headers=headers)
    return  json.loads(resp.text)

In [4]:
itsm = findItems('OGC API Features GRB')

members = itsm['member']
itsm ['totalItems']

3

In [ ]:
import os 

p = 
os.path.split(p.path)[-1]

NameError: name 'p' is not defined

In [13]:
url = _BASEURL +members[0]['@id']

resp = requests.get( url, headers=headers)

rec= json.loads(resp.text)
rec['catalogRecord']

{'identifier': 'fde20b42-c72c-47d0-b595-c82463359563',
 'url': 'https://metadata.vlaanderen.be/srv/dut/catalog.search#/metadata/fde20b42-c72c-47d0-b595-c82463359563',
 'modified': '2026-01-15T00:00:00Z',
 'primaryTopic': {'discriminator': 'DataService',
  '@type': 'DataService',
  'title': 'OGC API Features GRB',
  'description': 'Via de OGC API Features GRB kan je de vectordata van de verschillende objecten uit het Grootschalig Referentiebestand (GRB) opvragen. De OGC API Features GRB bevat alle GRB-gegevens gebaseerd op het GRBgis product. De gebruiker kan selecteren welke GRB-gegevens opgevraagd worden. Voor een gedetailleerde databeschrijving van het GRB raadpleegt u best het GRB-objectenhandboek via https://www.vlaanderen.be/digitaal-vlaanderen/onze-diensten-en-platformen/basiskaart-vlaanderen-grb/objectenhandboek-basiskaart-vlaanderen-grb',
  'modified': '2024-05-22T00:00:00Z',
  'license': None,
  'conformsTo': [{'title': 'Verordening (EG) n r. 976/2009 van de Commissie van 19 o

In [63]:
rec['catalogRecord']['primaryTopic']['distribution'][1]

{'title': 'OGC:WFS-2.0.0-http-get-feature',
 'description': None,
 'modified': None,
 'downloadUrl': None,
 'accessUrl': 'https://geoservice.waterinfo.be/OGRK/wfs?request=getfeature&service=wfs&version=1.1.0&typename=Overstromingsrisicokaarten-FLUVIAAL:lijninfrastructuren_FLU_noCC_T1000',
 'license': {'title': 'Modellicentie voor gratis hergebruik',
  'description': 'Onder deze licentie doet de instantie geen afstand van haar intellectuele rechten, maar mag de data voor eender welk doel hergebruikt worden, gratis en onder minimale restricties.',
  'url': 'https://data.vlaanderen.be/id/licentie/modellicentie-gratis-hergebruik/v1.0',
  'licenseType': [{'@type': 'Concept',
    'inscheme': 'http://purl.org/adms/licencetype/1.0',
    'label': 'Verplichte bronvermelding'}]},
 'rights': [],
 'format': None,
 'conformsTo': [{'title': None,
   'description': None,
   'type': None,
   'identifier': 'OGC:WFS-2.0.0-http-get-feature'}],
 'accessService': ['https://metadata.vlaanderen.be/srv/resourc